# Local training notebook: Step 3 on this PC

> **Aggregate, population-level language signals only. No diagnosis. No individual-level analysis.**

This is the **local-kernel twin** of the Step 3 part of `notebook.ipynb`. It trains and evaluates the same models on the **same data** as Colab:
`data/processed/dreaddit.parquet` plus the fixed split `data/processed/splits/dreaddit_split_v1.json`, which are also the files `colab_bundle.zip` ships to Colab.
Every step logs a fingerprint of those two files, so you can see that both kernels trained on identical data.

**Where things go.** Models: `trained_models/` in the project root (no Google Drive). Epoch checkpoints: `data/outputs/checkpoints/`, because the Trainer deletes old ones.

**Handoff and run-once guard.** Every step is logged with timestamps in `trained_models/HANDOFF.md` (readable) and `trained_models/handoff.json`.
Before a training step runs, the guard checks that log. If the step is already **done**, here or **on Colab**, it is skipped and the cell says where and when it ran.
For Colab's work to be visible here, import `colab_results.zip` first (cell L0.3). For the PC's work to be visible on Colab, rebuild `colab_bundle.zip` (cell L9) and re-upload it.

**Hardware.** The local Quadro P1000 has 4 GB of VRAM. The baseline (3A.1) is CPU-only and fine. Transformer training (3B.1, 3E.2) is slow here and may run out of memory at
batch 16. Cell L0.4 can trade per-step batch for gradient accumulation (same effective batch). Colab stays the faster option for 3B.1.

**Run All is safe.** Nothing trains unless you switch it on in `TRAIN_HERE` (cell L0.5). Units already done on either kernel are always skipped,
and evaluation cells print `NOT RUN …` until the models they need are here. Cells marked `USER RUNS THIS CELL` are the training cells.

Run order: **L0.1 → L0.2 → L0.3 → (L0.4) → L0.5 → L3.0a**, then the Step 3 cells, or just Run All.

In [ ]:
# Step L0.1: bootstrap (LOCAL kernel only). Imports the package, loads + validates config, seeds, versions.
import sys
from pathlib import Path

def _find_root(start: Path) -> Path:
    for c in (start, *start.parents):
        if (c / "code" / "config.yaml").is_file():
            return c
    raise FileNotFoundError("code/config.yaml not found above " + str(start))

PROJECT_ROOT = _find_root(Path.cwd().resolve())
SRC = str(PROJECT_ROOT / "code")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

from stress_signals import ETHICS_BANNER, __version__, get_logger, load_config, set_seed
from stress_signals import handoff as H
from stress_signals.utils import library_versions, models_root, require_local_kernel
require_local_kernel("notebook_local.ipynb")      # on Colab use notebook.ipynb

CFG = load_config(PROJECT_ROOT / "code" / "config.yaml")
LOG = get_logger("stress_signals", CFG["logging"]["level"], CFG["_paths"]["logs"] / "pipeline.log")
SEED = CFG["primary_seed"]
set_seed(SEED, deterministic=CFG["deterministic"])

print("=" * 88)
print(ETHICS_BANNER)
print("=" * 88)
print(f"stress_signals {__version__} | project root: {PROJECT_ROOT} | kernel: {H.kernel_name()}")
print(f"models + handoff log: {models_root(CFG)}")
print(f"training data fingerprint: {H.data_fingerprint(CFG)}")
for name, ver in library_versions().items():
    print(f"  {name:<13} {ver if ver else 'NOT INSTALLED'}")

In [ ]:
# Step L0.2: environment check (read-only). Tells you whether local GPU training is realistic.
from stress_signals.utils import detect_environment
ENV = detect_environment()
for key in ("python", "torch", "cuda_available", "gpu_name", "gpu_capability", "vram_total_gb", "vram_free_gb"):
    print(f"{key:<16} {ENV[key]}")
for w in ENV["warnings"]:
    print("  !", w)
print("\nlocal transformer training:", "GPU (slow on a P1000; see L0.4 for memory)" if ENV["cuda_available"]
      else "CPU only: expect hours per seed; prefer Colab for 3B.1")

In [ ]:
# Step L0.3: bring in what Colab trained, then show the handoff log (what ran where, and when).
from stress_signals.utils import require_local_kernel; require_local_kernel("Import of colab_results.zip")
# Looks for colab_results*.zip in the project root, data/outputs/ and your Downloads folder (newest wins). Existing files
# are never overwritten and nothing is deleted; the Colab log is merged. Importing the same zip twice does nothing.
H.import_colab_results(CFG)          # or: H.import_colab_results(CFG, r"C:\path\to\colab_results.zip")
H.show(CFG)

In [ ]:
# Step L0.4 (OPTIONAL): local memory settings for transformer training. Affects THIS kernel session only (config.yaml is
# untouched). Keeps the effective batch the same as Colab (train_batch_size x gradient_accumulation_steps = 16).
LOCAL_TRAIN_BATCH = None          # e.g. 4 on the 4 GB P1000 (-> 4 x 4 accumulation); None = same settings as Colab
sm = CFG["stress_model"]
if LOCAL_TRAIN_BATCH:
    effective = sm["train_batch_size"] * sm["gradient_accumulation_steps"]
    sm["train_batch_size"], sm["gradient_accumulation_steps"] = LOCAL_TRAIN_BATCH, max(1, effective // LOCAL_TRAIN_BATCH)
    sm["eval_batch_size"] = min(sm["eval_batch_size"], 2 * LOCAL_TRAIN_BATCH)
print(f"train batch {sm['train_batch_size']} x accumulation {sm['gradient_accumulation_steps']} "
      f"= effective {sm['train_batch_size'] * sm['gradient_accumulation_steps']} | eval batch {sm['eval_batch_size']}")

In [ ]:
# Step L0.5: WHAT MAY TRAIN ON THIS PC. Nothing trains unless you switch it on here, so "Run All" is always safe:
# a unit trains only if it is switched on below AND not already done on either kernel (handoff log). Everything else is
# skipped with a printed reason, and evaluation cells skip until the models they need exist.
TRAIN_HERE = {
    "baseline": False,     # True: fit the TF-IDF baseline here (CPU, ~1-3 min)
    "seeds": [],           # e.g. [42] or [42, 13, 2024]: transformer seeds to fine-tune here (slow on the P1000)
    "loso": False,         # True: the ~10 leave-one-subreddit-out trainings (3E.2) here
}
# Preview: what "Run All" would train right now
_mc = CFG["stress_model"]["model_choice"]
_units = [(H.BASELINE_STEP, TRAIN_HERE["baseline"])] + [(H.train_step_id(_mc, s), s in TRAIN_HERE["seeds"])
                                                          for s in CFG["stress_model"]["seeds"]]
for _step, _on in _units:
    _d = H.done_event(CFG, _step)
    print(f"{_step:<32}", f"done on {_d['kernel']} at {_d['utc']} -> skip" if _d else
          "WILL TRAIN HERE" if _on else "not done, not switched on -> skip")
print(f"{'3E.2/loso/*':<32}", "WILL TRAIN HERE (those not done yet)" if TRAIN_HERE["loso"] else "not switched on -> skip")

## Step 3: Stress-language classifier on Dreaddit (local kernel)
These cells are the same as in `notebook.ipynb`. Each training cell runs each unit of work **once across both kernels** (see the handoff log).
Evaluation cells only read saved predictions and can be re-run freely; they are logged as `ran`.

In [ ]:
# Step 3.0a: settings, transformers argument names, data guard (no training)
from stress_signals import stress_model as SM, metrics as MX

S3 = SM.settings(CFG)                       # pass a model name to switch, e.g. SM.settings(CFG, "distilroberta-base")
compat = SM.hf_compat()
print(f"model: {S3['model_choice']} -> {S3['hf_id']} | max_len {S3['max_length']} | lr {S3['learning_rate']} | "
      f"batch {S3['train_batch_size']} | epochs {S3['epochs']} | warmup {S3['warmup_ratio']} | seeds {S3['seeds']}")
print(f"transformers {compat['transformers']}: eval key = {compat['eval_strategy_key']!r}, "
      f"Trainer tokenizer arg = {compat['tokenizer_key']!r}")
D3 = SM.load_dreaddit(CFG)                  # verifies the saved split file and keeps text-only columns
for k, v in D3.items():
    print(f"{k:<10} n={len(v):>5}  stress share={v['label'].mean():.3f}  columns={list(v.columns)}")
print("runs will be written to:", SM.runs_root(CFG))
import pandas as pd

### 3A. Baseline: TF-IDF (word 1-2 + char_wb 2-5) with logistic regression

In [ ]:
# >>> USER RUNS THIS CELL. DO NOT EXECUTE. <<<
# Step 3A.1: fit the baseline (CPU is fine, ~1-3 min). Runs ONCE across both kernels: skipped when the handoff log says
# it is done (here, on the PC or on Colab) or when runs/baseline_tfidf_lr/DONE.json exists.
import pandas as pd
BASE_DIR = SM.runs_root(CFG) / "baseline_tfidf_lr"
H.run_step(CFG, H.BASELINE_STEP, "Fit TF-IDF + logistic regression baseline", lambda: SM.fit_baseline(CFG, D3),
           done_marker=BASE_DIR / "DONE.json", skip_result=BASE_DIR,
           allow=TRAIN_HERE["baseline"], blocked_reason=H.NOT_IN_TRAIN_HERE)
if (BASE_DIR / "c_selection_validation.csv").exists():
    print(pd.read_csv(BASE_DIR / "c_selection_validation.csv").round(4))

In [ ]:
# Step 3A.2: baseline metrics (validation + test, threshold 0.5, cluster-bootstrap 95% CIs). Reads saved predictions.
R3 = SM.readiness(CFG)
if H.ready(R3["baseline"], "3A.2", R3["why_baseline"]):
    BASE_EVAL = SM.evaluate_baseline(CFG)
    for split in ("validation", "test"):
        m, ci = BASE_EVAL[split]["all"], BASE_EVAL[split]["ci"]
        print(f"{split:<10} " + "  ".join(f"{k}={m[k]:.3f} [{ci[k]['lo']:.3f},{ci[k]['hi']:.3f}]"
                                          for k in ("f1_stress", "f1_macro", "roc_auc", "pr_auc", "accuracy")))
    H.log_ran(CFG, "3A.2/baseline_eval", "Evaluate baseline")

### 3B. Transformer fine-tuning
One run per seed, resume-safe. Seeds already done on Colab are skipped. Import `colab_results.zip` first (L0.3).

In [ ]:
# >>> USER RUNS THIS CELL. DO NOT EXECUTE. <<<
import json
# Step 3B.1: fine-tune all seeds. Each seed runs ONCE across both kernels: a seed the handoff log marks done (here, on
# the PC or on Colab) is skipped. Re-running after a disconnect resumes the unfinished seed from its last checkpoint.
MODEL_CHOICE = CFG["stress_model"]["model_choice"]     # "roberta-base" | "distilroberta-base" | "deberta-v3-base"
for seed in CFG["stress_model"]["seeds"]:
    rd = SM.run_dir_for(CFG, MODEL_CHOICE, seed)
    H.run_step(CFG, H.train_step_id(MODEL_CHOICE, seed), f"Fine-tune {MODEL_CHOICE}, seed {seed}",
               lambda seed=seed: SM.train_stress_seed(CFG, seed, MODEL_CHOICE), done_marker=rd / "DONE.json", skip_result=rd,
               allow=seed in TRAIN_HERE["seeds"], blocked_reason=H.NOT_IN_TRAIN_HERE)
    if (rd / "train_summary.json").exists():
        summ = json.loads((rd / "train_summary.json").read_text())
        print(f"seed {seed}: {rd} | best validation f1_stress = {summ.get('best_metric_validation_f1_stress')} "
              f"| epochs = {summ.get('epochs_completed')}")

In [ ]:
# Step 3B.2: training curves from the CSV logs (no training). Validation F1(stress) per epoch, per seed.
import json
MODEL_CHOICE = CFG["stress_model"]["model_choice"]
for seed, rd in SM.completed_seeds(CFG, MODEL_CHOICE).items():
    log = pd.read_csv(rd / "train_log.csv")
    ev = log[log["key"].isin(["eval_f1_stress", "eval_loss"])].pivot_table(index="epoch", columns="key", values="value")
    print(f"seed {seed}\n", ev.round(4))

### 3C. Evaluation (validation and official test, uncalibrated at threshold 0.5)
Needs every seed's predictions here, whether trained locally or imported from Colab.

In [ ]:
# Step 3C.1: per-seed metrics + CIs, mean +/- std across seeds (reads saved logits; no training)
R3 = SM.readiness(CFG, MODEL_CHOICE)
if H.ready(R3["all_seeds"], "3C.1", R3["why_seeds"]):
    EV3 = SM.evaluate_runs(CFG, MODEL_CHOICE)
    KEYS = ["accuracy", "precision_stress", "recall_stress", "f1_stress", "f1_macro", "roc_auc", "pr_auc"]
    for split in ("validation", "test"):
        rows = []
        for seed, r in EV3["seeds"].items():
            m, ci = r[split]["all"], r[split]["ci"]
            rows.append({"seed": seed, **{k: f"{m[k]:.3f} [{ci[k]['lo']:.3f},{ci[k]['hi']:.3f}]" for k in KEYS},
                         "confusion": m["confusion"]})
        print(f"\n== {split} (threshold 0.5, uncalibrated)"); display(pd.DataFrame(rows))
        print("mean +/- std across seeds:", {k: f"{v['mean']:.3f} ± {v['std']:.3f}" for k, v in EV3["summary"][split].items() if k in KEYS})
    H.log_ran(CFG, "3C.1/evaluate_seeds", "Evaluate all seeds (uncalibrated)")

In [ ]:
# Step 3C.2: high-confidence subset (same metrics restricted to confident annotations)
R3 = SM.readiness(CFG, MODEL_CHOICE)
if H.ready(R3["all_seeds"], "3C.2", R3["why_seeds"]):
    for split in ("validation", "test"):
        hc = {seed: r[split]["high_confidence"] for seed, r in EV3["seeds"].items()}
        first = next(iter(hc.values()))
        print(f"{split}: confidence >= {first['confidence_threshold']} keeps {first['share_of_split']:.1%} of rows (n={first['n']})")
        print("  mean +/- std:", {k: f"{v['mean']:.3f} ± {v['std']:.3f}" for k, v in EV3['summary'][f'{split}_high_confidence'].items() if k in KEYS})

In [ ]:
# Step 3C.3: per-subreddit error analysis (aggregates) + a small LOCAL error view (scrubbed, truncated, never saved)
R3 = SM.readiness(CFG, MODEL_CHOICE)
if H.ready(R3["all_seeds"], "3C.3", R3["why_seeds"]):
    FINAL_SEED = SM.select_final_seed(EV3)            # best VALIDATION f1_stress
    print("final seed (best validation f1_stress):", FINAL_SEED)
    display(SM.per_subreddit_errors(CFG, FINAL_SEED, "test", MODEL_CHOICE).round(3))
    from stress_signals.privacy import scrub_text
    te = SM.load_predictions(SM.run_dir_for(CFG, MODEL_CHOICE, FINAL_SEED), "test").merge(
        D3["test"][["record_id", "text_clean"]], on="record_id")
    te["error"] = (te["p"] >= 0.5).astype(int) != te["label"]
    errs = te[te["error"]].assign(margin=lambda d: (d["p"] - 0.5).abs()).sort_values("margin", ascending=False)
    nshow = min(CFG["privacy"]["max_error_examples_in_notebook"], 10)
    view = errs.head(nshow)[["community", "label", "p", "confidence"]].assign(
        text=errs.head(nshow)["text_clean"].map(lambda t: scrub_text(t, CFG)[:180]))
    display(view)   # most confident mistakes: label noise, sarcasm, or topic shortcuts?

### 3D. Calibration and threshold (fitted on VALIDATION only), then the calibrated test report once

In [ ]:
# Step 3D.1: fit T + threshold on validation; reliability diagram; ECE / Brier before vs after (validation)
R3 = SM.readiness(CFG, MODEL_CHOICE)
if H.ready(R3["all_seeds"], "3D.1", R3["why_seeds"]):
    CAL3 = SM.calibrate(CFG, FINAL_SEED, MODEL_CHOICE)
    dec = CAL3["threshold_decision"]; g = dec["validation_gain_f1_stress"]
    print(f"T = {CAL3['temperature']:.3f} | tuned threshold (searched {CAL3['threshold_search_grid'][:2]}) = {CAL3['threshold_tuned']:.2f}")
    print(f"validation F1(stress) gain over 0.5 = {g['gain']:+.4f} [95% CI {g['lo']:+.4f}, {g['hi']:+.4f}] "
          f"-> APPLIED threshold = {CAL3['threshold']:.2f} ({dec['reason']})")
    for k in ("before", "after"):
        v = CAL3[f"validation_{k}"]
        print(f"validation {k:<6}: ECE {v['ece']:.4f} | Brier {v['brier']:.4f} | F1(stress) {v['f1_stress']:.3f} @ {v['threshold']:.2f}")
    print(f"validation after, at tuned {CAL3['threshold_tuned']:.2f}: F1(stress) {CAL3['validation_after_tuned']['f1_stress']:.3f}")
    from IPython.display import Image
    display(Image(str(SM.run_dir_for(CFG, MODEL_CHOICE, FINAL_SEED) / "reliability_validation.png")))
    H.log_ran(CFG, "3D.1/calibrate", "Calibrate final seed on validation")

In [ ]:
# Step 3D.2: FINAL calibrated test report: computed ONCE (FINAL_TEST_REPORT.json is reused afterwards).
# Reported at BOTH thresholds; "applied" is the one the decision rule picked on validation.
R3 = SM.readiness(CFG, MODEL_CHOICE)
if H.ready(R3["all_seeds"], "3D.2", R3["why_seeds"]):
    FIN3 = SM.final_test_report(CFG, FINAL_SEED, MODEL_CHOICE)
    print(f"TEST (T={FIN3['temperature']:.3f}); applied threshold = {FIN3['threshold']:.2f} [{FIN3['threshold_applied']}]: "
          f"{FIN3['threshold_decision_reason']}")
    rows = []
    for name in ("at_threshold_0.5", "at_threshold_tuned"):
        b = FIN3[name]
        for k in KEYS + ["brier"]:
            rows.append({"threshold": f"{b['threshold']:.2f}", "metric": k, "value": round(b["test"][k], 3),
                         "ci95": f"[{b['test_ci'][k]['lo']:.3f}, {b['test_ci'][k]['hi']:.3f}]" if k in b["test_ci"] else ""})
    display(pd.DataFrame(rows).pivot(index="metric", columns="threshold", values=["value", "ci95"]))
    print("n =", FIN3["test"]["n"], "| ECE (threshold-free)", round(FIN3["test_ece"], 4))
    for name in ("at_threshold_0.5", "at_threshold_tuned"):
        b = FIN3[name]
        print(f"  @{b['threshold']:.2f} confusion {b['test']['confusion']} | high-confidence F1(stress) "
              f"{b['test_high_confidence']['f1_stress']:.3f} (n={b['test_high_confidence']['n']})")
    H.log_ran(CFG, "3D.2/final_test_report", "Final calibrated test report")

### 3E. Shortcut-learning checks

In [ ]:
# Step 3E.1: topic masking vs random-masking control (5 draws) on the official test set (INFERENCE ONLY; GPU or CPU)
# Uses the APPLIED threshold from 3D. Diagnostic only: nothing is tuned on these numbers.
R3 = SM.readiness(CFG, MODEL_CHOICE)
if H.ready(R3["all_seeds"], "3E.1", R3["why_seeds"]):
    model_dir = SM.run_dir_for(CFG, MODEL_CHOICE, FINAL_SEED) / "best_model"
    SHORT = SM.shortcut_masking_eval(CFG, model_dir, "test", CAL3["temperature"], CAL3["threshold"], MODEL_CHOICE)
    display(SHORT[["condition", "draw_seed", "f1_stress", "roc_auc", "accuracy", "predicted_positive_rate"]].round(3))
    print(f"texts with >=1 masked term: {SHORT['texts_with_masked_terms'].iloc[0]} of {SHORT['n'].iloc[0]}; "
          f"terms masked: {SHORT['masked_terms_total'].iloc[0]}")
    display(SM.summarise_masking(SHORT).round(4))
    H.log_ran(CFG, "3E.1/topic_masking", "Topic-masking shortcut check")

In [ ]:
# >>> USER RUNS THIS CELL. DO NOT EXECUTE. <<<
# Step 3E.2 (OPTIONAL): leave-one-subreddit-out. Trains one model per held-out subreddit (distilroberta by default,
# 1 seed, config stress_model.shortcut.loso_*). ~10 trainings: run on a GPU. Each subreddit is one handoff step, so it
# runs ONCE across both kernels; resume-safe like 3B.1.
import pandas as pd
LOSO_MODEL = CFG["stress_model"]["shortcut"]["loso_model"]
for sub in sorted(pd.concat(D3.values())["community"].unique()):
    d = SM.runs_root(CFG) / "loso" / f"{LOSO_MODEL}_{sub}"
    H.run_step(CFG, H.loso_step_id(LOSO_MODEL, sub), f"LOSO: train {LOSO_MODEL} without r/{sub}",
               lambda sub=sub: SM.train_loso(CFG, [sub])[sub], done_marker=d / "DONE.json", skip_result=d,
               allow=TRAIN_HERE["loso"], blocked_reason=H.NOT_IN_TRAIN_HERE)
LOSO = SM.evaluate_loso(CFG)
if not len(LOSO):
    print("no finished LOSO runs on this kernel yet")
elif "FINAL_SEED" in globals():                        # in-domain comparison needs 3C.3 (local evaluation)
    inorig = SM.per_subreddit_errors(CFG, FINAL_SEED, "test", MODEL_CHOICE)[["community", "n", "f1_stress", "roc_auc"]]
    display(LOSO.merge(inorig, left_on="heldout_subreddit", right_on="community", suffixes=("_heldout", "_in_domain_test")).round(3))
else:
    display(LOSO.round(3))

In [ ]:
# Step 3E.3: baseline n-grams with the largest weights (needs 3A.1). topic_term=True = contains a subreddit/domain term.
R3 = SM.readiness(CFG)
if H.ready(R3["baseline"], "3E.3", R3["why_baseline"]):
    NG = SM.top_ngrams(CFG)
    for direction in ("stress", "not_stress"):
        sub = NG[(NG["direction"] == direction) & (NG["type"] == "word")]
        print(f"\n{direction}: share of top word n-grams that are topic terms = {sub['topic_term'].mean():.2f}")
        display(sub[["rank", "ngram", "coef", "topic_term"]].head(20))

### 3F. Save the model bundle
Writes `trained_models/stress/v<YYYYMMDD>/` once (see `notebook.ipynb` 3F for the contents).

In [ ]:
# Step 3F: write the bundle (no training), ONCE across both kernels. Run after 3D.2 (and optionally 3A, 3E.1, 3E.2 so
# metrics.json includes them). Output: trained_models/stress/v<YYYYMMDD>/.
R3 = SM.readiness(CFG, MODEL_CHOICE)
if H.ready(R3["all_seeds"] and "FINAL_SEED" in globals() and (SM.run_dir_for(CFG, MODEL_CHOICE, FINAL_SEED) / "FINAL_TEST_REPORT.json").exists(), "3F", R3["why_seeds"]):
    BUNDLE3 = H.run_step(CFG, H.BUNDLE_STEP, f"Save stress model bundle ({MODEL_CHOICE}, seed {FINAL_SEED})",
                         lambda: SM.save_bundle(CFG, FINAL_SEED, MODEL_CHOICE))
    if BUNDLE3 is None:                                    # skipped: show the bundle the log points to
        ev = H.done_event(CFG, H.BUNDLE_STEP)
        BUNDLE3 = PROJECT_ROOT / ev["outputs"][0] if ev and ev["outputs"] else None
    print("bundle:", BUNDLE3)
    if BUNDLE3 is not None and BUNDLE3.is_dir():
        for f in sorted(BUNDLE3.iterdir()):
            print(f"  {f.name:<28} {f.stat().st_size/1e6:8.2f} MB")
        print((BUNDLE3 / "model_card.md").read_text()[:1500])

### L9. Hand off to Colab and review the log
Rebuild `colab_bundle.zip` after local training. It carries this PC's handoff log, so Colab skips the seeds already trained here. Re-upload it and re-run 0.0 on Colab.
The full timestamped history is in `trained_models/HANDOFF.md`.

In [ ]:
# Step L9: share local progress with Colab + show the handoff log
from stress_signals.utils import build_colab_bundle
B = build_colab_bundle(CFG)
print(B, f"{B.stat().st_size/1e6:.2f} MB  -> right-click > Upload to Colab, then run 0.0 on the Colab kernel")
H.show(CFG)